# SFT Training — Hafta 3

**Proje:** Knowledge Distillation & Reasoning-based Data Augmentation
**Faz:** Student modeli teacher dataset ile QLoRA fine-tune

---

## Tasarım

Bu notebook **tek bir konfigürasyon ile bir SFT run'ı** yapar.
Her run için sadece `SPLIT_NAME` ve `DATASET_REPO` değiştirilir.

### v2 Runları (GT-seeded — tamamlandı)
| Run | Split | Sonuç |
|---|---|---|
| **v2-Dense** | `dense` | %66 ✓ |
| **v2-Scale** | `scale_baseline` | %62 ✓ |

### v3 Runları (Qwen-generated — şu an)
| Run | Split | Hedef |
|---|---|---|
| **v3-Dense** | `dense_qwen_1000` | v2-Dense ile karşılaştır |
| **v3-Scale** | `scale_qwen_6000` | v2-Scale ile karşılaştır |

**Araştırma sorusu:** GT-seeded vs Qwen-generated çözümler — hangisi daha iyi öğretir?

---

## Student Model

Hafta 1 baseline'a göre top adaylar:
- **Qwen3.5-4B** — primary (56% baseline, en iyi size/perf)
- **Qwen3-8B** — secondary (69% baseline, ceiling testi)

Bu notebook varsayılan olarak **Qwen3.5-4B** kullanır.

## 1. Kurulum

In [ ]:
# 1. Hücre - Yeni
import os
from google.colab import drive
drive.mount('/content/drive')

!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

!mkdir -p /content/datasets
!kaggle datasets download -d sewowashere/soru-uzerinde-diyaloglar
!unzip soru-uzerinde-diyaloglar.zip -d /content/datasets/
print("✓ Datasetler /content/datasets altına indirildi.")

Mounted at /content/drive
cp: cannot stat 'kaggle.json': No such file or directory
chmod: cannot access '/root/.kaggle/kaggle.json': No such file or directory
Dataset URL: https://www.kaggle.com/datasets/sewowashere/soru-uzerinde-diyaloglar
License(s): unknown
100% 4.04M/4.04M [00:01<00:00, 2.55MB/s]

Archive:  soru-uzerinde-diyaloglar.zip
  inflating: /content/datasets/gsm8k_gemini_2_5_sonuclari-1000.json  
  inflating: /content/datasets/gsm8k_gemini_2_5_sonuclari-farkli-yaklasimla-cozum.json  
  inflating: /content/datasets/gsm8k_gemini_2_5_sonuclari-kritik-nokta.json  
  inflating: /content/datasets/gsm8k_gemini_2_5_sonuclari-neler-bilmek-gerekir (tam).json  
  inflating: /content/datasets/gsm8k_gemini_2_5_sonuclari-tuzaklar-1000.json  
  inflating: /content/datasets/gsm8k_gemini_2_5_sonuclari-tuzaklar-2000.json  
✓ Datasetler /content/datasets altına indirildi.


In [ ]:
# GPU kontrolü
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
import torch
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

NVIDIA L4, 23034 MiB
CUDA: True
GPU: NVIDIA L4
VRAM: 23.7 GB


In [ ]:
%%capture
# Qwen3.5 desteği için transformers'in main'i şart
!pip install -q git+https://github.com/huggingface/transformers.git
!pip install -q -U trl peft accelerate bitsandbytes datasets huggingface_hub
!pip install -q kagglehub[pandas-datasets]

## 2. Konfigürasyon

**Bu hücreyi her run için değiştir:**
- `RUN_NAME` ve `SPLIT_NAME` → A/B/C senaryoları için
- Diğer hyperparam'lar 3 run'da aynı kalır (fair comparison)

In [ ]:
# ──────────────────────────────────────────────────────────────────────────
# RUN KONFİGÜRASYONU — bu hücreyi her run için düzenle
# ──────────────────────────────────────────────────────────────────────────

# ── SPLIT_NAME & DATASET_REPO seçenekleri ────────────────────────────────
#
# [v2 — GT-seeded, tamamlandı]
#   DATASET_REPO = "AliMertTemizsoy/gsm8k-tr-dense-vs-scale-qwen-plus"
#   SPLIT_NAME   = "dense"            → adapter: qwen3_5-4b-dense           (%66)
#   SPLIT_NAME   = "scale_baseline"   → adapter: qwen3_5-4b-scale_baseline  (%62)
#
# [v3 — Qwen-generated, tamamlandı]
#   DATASET_REPO = "AliMertTemizsoy/gsm8k-tr-qwen-v3"
#   SPLIT_NAME   = "dense_qwen_1000"  → adapter: qwen3_5-4b-dense_qwen_1000 (%70)
#   SPLIT_NAME   = "scale_qwen_6000"  → adapter: qwen3_5-4b-scale_qwen_6000 (%63)
#
# [Ablation — meta-soru sayısı]
#   DATASET_REPO = "AliMertTemizsoy/gsm8k-tr-qwen-ablation"
#   SPLIT_NAME   = "dense_qwen_2"     → 1000×2 = 2000 instance
#   SPLIT_NAME   = "scale_qwen_2000"  → 2000 plain (dense_2 karşılaştırması)
#   SPLIT_NAME   = "dense_qwen_4"     → 1000×4 = 4000 instance
#   SPLIT_NAME   = "scale_qwen_4000"  → 4000 plain (dense_4 karşılaştırması)
#
# [Gemini — Şevval]
#   DATASET_REPO = "<HF_USERNAME>/gsm8k-tr-gemini-v1"
#   SPLIT_NAME   = "dense_gemini_1000"
#   SPLIT_NAME   = "scale_gemini_6000"
#
# ─────────────────────────────────────────────────────────────────────────

SPLIT_NAME   = "dense_qwen_2"                              # <-- DUZENLE
DATASET_REPO = "AliMertTemizsoy/gsm8k-tr-qwen-ablation"   # <-- DUZENLE

# Run identifier (output klasörü ve HF push için)
RUN_NAME = f"qwen3_5-4b-{SPLIT_NAME}"  # otomatik

# Student model
STUDENT_MODEL = "Qwen/Qwen3.5-4B"  # <-- alternatif: "Qwen/Qwen3-8B"

# ── Hyperparameters (3 run da aynı) ──────────────────────────────────────
NUM_EPOCHS = 3
LEARNING_RATE = 2e-4
PER_DEVICE_BATCH_SIZE = 2
GRAD_ACCUM_STEPS = 8        # effective batch = 2 × 8 = 16
MAX_SEQ_LENGTH = 2048       # multi-turn için yeterli, scale için fazla
WARMUP_RATIO = 0.05
WEIGHT_DECAY = 0.01

# LoRA config
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05

# Training mode: assistant turn'lerinde loss
# True → student sadece teacher'ın "asistan cevaplarını" öğrenir, user mesajları masklenir
ASSISTANT_ONLY_LOSS = False  # Qwen3.5 chat template ile uyumsuz, kapatıldı

# Logging / saving
LOGGING_STEPS = 5
SAVE_STEPS = 100
EVAL_STEPS = 50

# Output
from pathlib import Path
OUTPUT_DIR = Path(f"/content/drive/MyDrive/sft_outputs_retrain/{RUN_NAME}_retrain")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Run         : {RUN_NAME}")
print(f"Split       : {SPLIT_NAME}")
print(f"Student     : {STUDENT_MODEL}")
print(f"Output dir  : {OUTPUT_DIR}")

Run         : qwen3_5-4b-dense_qwen_2
Split       : dense_qwen_2
Student     : Qwen/Qwen3.5-4B
Output dir  : /content/drive/MyDrive/sft_outputs_retrain/qwen3_5-4b-dense_qwen_2_retrain


In [ ]:
import os
import json
import random
import torch
import numpy as np
from datetime import datetime
import pandas as pd
import kagglehub
from kagglehub import KaggleDatasetAdapter

from datasets import load_dataset, Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig

# Reproducibility
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# HF token (model gating + push için)
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    HF_TOKEN = os.environ.get("HF_TOKEN", "")

print(f"HF token: {'OK' if HF_TOKEN else 'YOK!'}")

HF token: OK


## 3. Dataset Yükle

HF'deki dataset'te `messages` alanı **JSON string** olarak saklı (push sırasında stringify edildi).
Geri parse edip `list[dict]` formatına çeviriyoruz — TRL bu formatı bekler.

In [ ]:
KAGGLE_DATASET = "sewowashere/soru-uzerinde-diyaloglar"

KAGGLE_FILES = [
    "gsm8k_gemini_2_5_sonuclari-1000.json",
    "gsm8k_gemini_2_5_sonuclari-farkli-yaklasimla-cozum.json",
    "gsm8k_gemini_2_5_sonuclari-kritik-nokta.json",
    "gsm8k_gemini_2_5_sonuclari-neler-bilmek-gerekir (tam).json",
    "gsm8k_gemini_2_5_sonuclari-tuzaklar-1000.json",
    "gsm8k_gemini_2_5_sonuclari-tuzaklar-2000.json",
]

all_examples = []

for file_name in KAGGLE_FILES:

    print(f"\nYükleniyor: {file_name}")

    df = kagglehub.load_dataset(
        KaggleDatasetAdapter.PANDAS,
        KAGGLE_DATASET,
        file_name,
    )

    print(f"{len(df)} kayıt bulundu")

    for _, row in df.iterrows():

        messages = [
            {
                "role": "user",
                "content": str(row["question"])
            },
            {
                "role": "assistant",
                "content": str(row["model_output"])
            }
        ]

        all_examples.append({
            "messages": messages
        })

train_ds = Dataset.from_list(all_examples)

split = train_ds.train_test_split(
    test_size=0.05,
    seed=SEED
)

train_ds = split["train"]
eval_ds = split["test"]

print(f"Train: {len(train_ds)}")
print(f"Eval : {len(eval_ds)}")

print(train_ds[0])


Yükleniyor: gsm8k_gemini_2_5_sonuclari-1000.json


/tmp/ipykernel_10062/817272856.py:18: DeprecationWarning: Use dataset_load() instead of load_dataset(). load_dataset() will be removed in a future version.
  df = kagglehub.load_dataset(


Using Colab cache for faster access to the 'soru-uzerinde-diyaloglar' dataset.
1000 kayıt bulundu

Yükleniyor: gsm8k_gemini_2_5_sonuclari-farkli-yaklasimla-cozum.json


/tmp/ipykernel_10062/817272856.py:18: DeprecationWarning: Use dataset_load() instead of load_dataset(). load_dataset() will be removed in a future version.
  df = kagglehub.load_dataset(


Using Colab cache for faster access to the 'soru-uzerinde-diyaloglar' dataset.
860 kayıt bulundu

Yükleniyor: gsm8k_gemini_2_5_sonuclari-kritik-nokta.json


...

Using Colab cache for faster access to the 'soru-uzerinde-diyaloglar' dataset.
1000 kayıt bulundu

Yükleniyor: gsm8k_gemini_2_5_sonuclari-tuzaklar-1000.json


/tmp/ipykernel_10062/817272856.py:18: DeprecationWarning: Use dataset_load() instead of load_dataset(). load_dataset() will be removed in a future version.
  df = kagglehub.load_dataset(


Using Colab cache for faster access to the 'soru-uzerinde-diyaloglar' dataset.
1000 kayıt bulundu

Yükleniyor: gsm8k_gemini_2_5_sonuclari-tuzaklar-2000.json


/tmp/ipykernel_10062/817272856.py:18: DeprecationWarning: Use dataset_load() instead of load_dataset(). load_dataset() will be removed in a future version.
  df = kagglehub.load_dataset(


Using Colab cache for faster access to the 'soru-uzerinde-diyaloglar' dataset.
2000 kayıt bulundu
Train: 6517
Eval : 343
{'messages': [{'role': 'user', 'content': "Adam işinden günlük 40 dolar kazanıyor. Parasının yüzde 10'u vergi olarak kesiliyor. Adam 30 gün çalıştıktan sonra vergilerden sonra ne kadar para kazanacak?"}, {'role': 'assistant', 'content': 'Elbette, sorunun çözümü ve analizi aşağıdadır.\n\n**Çözüm:**\n\n1.  **Toplam Brüt Kazancın Hesaplanması:**\n    Adam günde 40 dolar kazanıyor ve 30 gün çalışıyor.\n    Toplam brüt kazanç = Günlük kazanç × Çalışılan gün sayısı\n    Toplam brüt kazanç = 40 dolar/gün × 30 gün = 1200 dolar\n\n2.  **Vergi Kesintisinin Hesaplanması:**\n    Vergi, toplam brüt kazancın %10\'u olarak kesiliyor.\n    Toplam vergi kesintisi = Toplam brüt kazanç × Vergi oranı\n    Toplam vergi kesintisi = 1200 dolar × %10 = 1200 × (10/100) = 120 dolar\n\n3.  **Net Kazancın Hesaplanması:**\n    Vergilerden sonra ele geçecek para (net kazanç), toplam brüt kazançta

## 4. Tokenizer & Model (4-bit QLoRA)

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(
    STUDENT_MODEL, token=HF_TOKEN, trust_remote_code=True,
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"  # SFT için right padding daha iyi

# Chat template kontrolü
print(f"Chat template var mı: {tokenizer.chat_template is not None}")
if tokenizer.chat_template:
    test_msg = train_ds[0]["messages"][:2]
    formatted = tokenizer.apply_chat_template(test_msg, tokenize=False, add_generation_prompt=False)
    print(f"\nFormatlama örneği (ilk 300 char):\n{formatted[:300]}")

config.json:   0%|          | 0.00/3.16k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/16.7k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/6.72M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/3.35M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/12.8M [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/7.76k [00:00<?, ?B/s]

Chat template var mı: True

Formatlama örneği (ilk 300 char):
<|im_start|>user
Adam işinden günlük 40 dolar kazanıyor. Parasının yüzde 10'u vergi olarak kesiliyor. Adam 30 gün çalıştıktan sonra vergilerden sonra ne kadar para kazanacak?<|im_end|>
<|im_start|>assistant
<think>

</think>

Elbette, sorunun çözümü ve analizi aşağıdadır.

**Çözüm:**

1.  **Toplam B


In [ ]:
# 4-bit NF4 quantization (QLoRA setup, Hafta 1 baseline ile aynı)
compute_dtype = torch.bfloat16 if torch.cuda.get_device_capability()[0] >= 8 else torch.float16

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=compute_dtype,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    STUDENT_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
    token=HF_TOKEN,
    trust_remote_code=True,
    torch_dtype=compute_dtype,
)
model.config.use_cache = False  # gradient checkpointing ile uyumsuzluk olmasın
model.config.pretraining_tp = 1

# LoRA için hazırla (k-bit training)
model = prepare_model_for_kbit_training(
    model, use_gradient_checkpointing=True,
)

print(f"Model yüklendi: {STUDENT_MODEL}")
print(f"VRAM kullanımı: {torch.cuda.memory_allocated()/1e9:.1f} GB")

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json:   0%|          | 0.00/76.2k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

[transformers] The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d


Loading weights:   0%|          | 0/426 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Model yüklendi: Qwen/Qwen3.5-4B
VRAM kullanımı: 4.4 GB


In [ ]:
# LoRA config — tüm linear layer'lara uygula
lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules="all-linear",  # Qwen3 için tüm projection'lar
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 32,464,896 || all params: 4,238,216,192 || trainable%: 0.7660


## 5. SFT Trainer

TRL'in `SFTTrainer`'ı multi-turn `messages` formatını otomatik handle eder.
`assistant_only_loss=True` ile **sadece assistant turn'lerinde loss hesaplanır** — user mesajları
ve system prompt loss'a katkı vermez. Bu özellikle multi-turn dense dataset için önemli.

In [ ]:
sft_config = SFTConfig(
    output_dir=str(OUTPUT_DIR),
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=PER_DEVICE_BATCH_SIZE,
    per_device_eval_batch_size=PER_DEVICE_BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM_STEPS,
    learning_rate=LEARNING_RATE,
    warmup_steps=50,  # warmup_ratio deprecated, sabit step kullan
    weight_decay=WEIGHT_DECAY,
    lr_scheduler_type="cosine",
    optim="paged_adamw_8bit",  # bellek dostu
    bf16=(compute_dtype == torch.bfloat16),
    fp16=False,  #(compute_dtype == torch.float16),
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    max_length=MAX_SEQ_LENGTH,
    packing=False,             # multi-turn için packing kapalı (bağlam korunur)
    assistant_only_loss=ASSISTANT_ONLY_LOSS,
    logging_steps=LOGGING_STEPS,
    save_steps=SAVE_STEPS,
    save_total_limit=2,
    eval_strategy="steps",
    eval_steps=EVAL_STEPS,
    report_to="none",          # wandb istemiyorsak
    seed=SEED,
    dataloader_num_workers=2,
    remove_unused_columns=True,
)

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    processing_class=tokenizer,
)

print("Trainer hazır.")

Tokenizing train dataset:   0%|          | 0/6517 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/343 [00:00<?, ? examples/s]

Trainer hazır.


## 6. Training

**Tahmini süre (A100 40GB, Qwen3.5-4B QLoRA):**
- `dense` (1000 multi-turn): ~1-2 saat (3 epoch)
- `solution_only` (1000 single-turn): ~30-45 dk
- `scale_baseline` (6000 single-turn): ~3-4 saat

Loss düştükçe progress bar'da göreceksin. Eğer eval loss artmaya başlarsa overfitting var demektir,
epoch sayısını düşürmek gerekebilir.

In [ ]:
train_result = trainer.train()
print("\nEğitim tamamlandı!")
print(f"Final loss: {train_result.training_loss:.4f}")

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 248046, 'pad_token_id': 248044}.


Step,Training Loss,Validation Loss


Step,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
50,0.815612,0.808164,0.839920,483976.000000,0.788064


In [ ]:
from google.colab import drive

drive.mount('/content/drive')

SAVE_DIR = "/content/drive/MyDrive/qwen3_5_4b_lora_final"

trainer.save_model(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)

print(f"Model kaydedildi: {SAVE_DIR}")

with open(f"{SAVE_DIR}/train_metrics.json", "w") as f:
    json.dump({
        "run_name": RUN_NAME,
        "student": STUDENT_MODEL,
        "final_loss": float(train_result.training_loss),
        "epochs": NUM_EPOCHS,
        "timestamp": datetime.now().isoformat(),
    }, f, indent=2)

print("Metrics kaydedildi.")

## 7. Hızlı Sanity Check

Eğitilmiş modelden 1-2 örnek üret, çalıştığını doğrula.

In [ ]:
from transformers import pipeline

model.eval()

test_questions = [
    "Ali'nin 5 elması, Ayşe'nin ise Ali'nin 3 katı kadar elması var. Toplam kaç elma vardır?",
    "Bir kitap 25 TL. Mert 4 kitap aldıysa ve 10 TL indirim aldıysa kaç TL ödedi?",
]

for q in test_questions:
    msgs = [
        {"role": "user", "content": f"Aşağıdaki matematik sorusunu adım adım çözer misin? Son cevabı '#### [Sayı]' formatında belirt.\n\nSoru: {q}"}
    ]
    prompt = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=400,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    text = tokenizer.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    print(f"Q: {q}")
    print(f"A: {text[:500]}...\n{'-'*70}")

In [ ]:
#bildirim gönderme
import requests
import json

def send_push_alert(title, message):
    # Uygulamanın sana verdiği API Key'i buraya yapıştır
    api_key = "3193d93e-6601-4ea7-995f-f693a6a1139d"

    # Dökümantasyondaki doğru URL yapısı:
    url = f"https://push.techulus.com/api/v1/notify/{api_key}"

    payload = {
        "title": title,
        "body": message
    }

    headers = {
        "Content-Type": "application/json"
    }

    try:
        # POST isteğini gönderiyoruz
        response = requests.post(url, headers=headers, data=json.dumps(payload), timeout=10)

        if response.status_code == 200:
            print("Bildirim başarıyla gönderildi! ✅")
        else:
            print(f"Hata Kodu: {response.status_code}")
            print(f"Sunucu Yanıtı: {response.text}")

    except Exception as e:
        print(f"Bir bağlantı hatası oluştu: {e}")

# Test etmek için çalıştır:
send_push_alert("kardeş eğitim bitti", "git bilgisayarı kontrol et.")

## 8. HF'ye Adapter Push

Adapter'ı HF'ye yükle — Hafta 4 değerlendirme notebook'u oradan çekecek.

In [ ]:
HF_REPO_ADAPTER = f"AliMertTemizsoy/{RUN_NAME}"  # <-- DUZENLE (kullanıcı adın doğru mu?)

model.push_to_hub(HF_REPO_ADAPTER, token=HF_TOKEN, private=True)
tokenizer.push_to_hub(HF_REPO_ADAPTER, token=HF_TOKEN, private=True)

print(f"Push edildi: https://huggingface.co/{HF_REPO_ADAPTER}")

## Sonraki Adım

Bu notebook'u **3 kez** çalıştır (config hücresinde sadece `SPLIT_NAME` değişir):

1. `SPLIT_NAME = "dense"` → adapter A
2. `SPLIT_NAME = "solution_only"` → adapter B
3. `SPLIT_NAME = "scale_baseline"` → adapter C

Sonra Hafta 4 evaluation notebook'unda 3 adapter'ı + Hafta 1 baseline'ı (raw model) GSM8K-TR
**test split** üzerinde karşılaştır.

**Beklenen sıralama:** A > C > B > raw baseline